# Časť 2, Téma 1: Úvod do vkladania chýb do napájania (Voltage Glitching) (MAIN)

---
POZNÁMKA: Toto praktické cvičenie odkazuje na niektoré (komerčné) školiace materiály na webe [ChipWhisperer.io](https://www.ChipWhisperer.io). Cvičenie môžete voľne spustiť a používať v súlade s licenciou open-source (vrátane jeho použitia vo vašich vlastných kurzoch, ak ich budete distribuovať podobným spôsobom), musíte však zachovať informáciu o tomto pôvodnom zdroji. Ak si chcete vyskúšať plnohodnotný zážitok, zvážte účasť na našom školení.

---

**ZHRNUTIE:** *Podobne ako pri glitchovaní hodinového signálu (clock glitching), aj vkladanie krátkych chýb do napájacej línie embedded zariadenia môže viesť k preskočeniu inštrukcií a poškodeniu výsledkov. Okrem toho, že voltage glitching poskytuje na niektorých cieľoch spoľahlivejšie výsledky v porovnaní s clock glitchingom, má tiež výhodu v tom, že napájacie piny (Vcc) na čipoch bývajú oveľa lepšie prístupné. V tomto kurze sa tomu nebudeme podrobne venovať, ale táto metóda sa dá použiť aj na glitchovanie zariadenia asynchrónne k jeho hodinovému signálu.*

**VÝSLEDKY VZDELÁVANIA:**

* Pochopenie nastavení pre vkladanie chýb do napájania (voltage glitch)
* Vytvorenie mapy úspešných glitchov a pádov (crash map)
* Úprava glitchovacieho obvodu za účelom zvýšenia úspešnosti útokov

## Hardvér pre Voltage Glitching

ChipWhisperer využíva rovnaký hardvérový blok pre napäťové aj hodinové glitchovanie. Jediný rozdiel spočíva v tom, kam je smerovaný výstup chyby. Namiesto smerovania na pin HS2 sa napäťové glitchovanie vykonáva presmerovaním chyby buď na tranzistor `glitch_hp` (high power), alebo na tranzistor `glitch_lp` (low power). To je možné vykonať pomocou nasledujúcich volaní API:

```python
scope.io.glitch_hp = True #povoliť HP glitch
scope.io.glitch_hp = False #zakázať HP glitch
scope.io.glitch_lp = True #povoliť LP glitch
scope.io.glitch_lp = False #zakázať LP glitch
```

Hoci sú hardvérové bloky rovnaké, budete musieť zmeniť spôsob ich konfigurácie. Určite by ste nechceli smerovať výstup `"clock_xor"` na glitchovací tranzistor a rozkmitať Vcc rovnako ako hodinový signál zariadenia! Namiesto toho sú pre napäťové glitchovanie najvhodnejšie tieto dve nastavenia výstupu:

1. `"glitch_only"` - vloží chybu len na časť hodinového cyklu na základe hodnôt `scope.glitch.width` a `scope.glitch.offset`
1. `"enable_only"` - vloží chybu na celý hodinový cyklus

Nastavenie `"enable_only"` bude pre väčšinu zariadení zvyčajne príliš silné. Jednou zo situácií, kedy prekonáva `"glitch_only"`, je glitchovanie asynchrónne k hodinám cieľa. Príkladom je glitchovanie cieľového zariadenia, ktoré využíva interný oscilátor. V takom prípade môže byť hodinový signál ChipWhispereru zrýchlený vysoko nad frekvenciu cieľa, aby sa vložila presná chyba, pričom parameter `repeat` funguje ako šírka (`width`) a `ext_offset` funguje ako posun (`offset`).

### Napäťové glitchovanie (Voltage Glitching) vs. Hodinové glitchovanie (Clock Glitching)

Napäťové glitchovanie prináša oproti hodinovému zjavné výhody, ako je napríklad funkčnosť na širšom spektre cieľových zariadení, avšak jeho nevýhody sú menej zrejmé. Jednou z najväčších nevýhod je, ako veľmi závisí od samotného fyzického glitchovacieho obvodu. Pri hodinovom glitchovaní je relatívne jednoduché vložiť chybu – neexistuje tam žiadny externý prvok, ktorý by sa snažil udržať hodinovú líniu na určitej napäťovej úrovni. Pri napájacích pinoch cieľa je to úplne inak. Keď sa pokúsime stiahnuť napájací pin k zemi (GND), bojuje proti nám množstvo komponentov, ktoré sa snažia udržať na pine správne napätie – napríklad dekuplačné kondenzátory, filtračné kondenzátory napájacieho zdroja a samotný regulátor napätia. To znamená, že keď urobíme aj drobné zmeny v glitchovacom obvode, nastavenia chýb a dokonca aj naša schopnosť vôbec nejakú chybu vyvolať sa môžu úplne zmeniť! Predstavte si glitchovanie cieľa na doske UFO CW308. Ak vymeníte koaxiálny kábel s dĺžkou 20 cm za 40 cm kábel, budete musieť nájsť úplne nové nastavenia, aby ste útok zopakovali (ak to vôbec ešte bude možné). To sa dá veľmi ľahko overiť na osciloskope alebo pomocou ADC na ChipWhispereri: dlhšie káble a bočníky (shunt odpory) s nižšou hodnotou spôsobia, že hrany glitchu budú menej ostré a zvýši sa zvonenie signálu (ringing).

Hoci vašou prvou myšlienkou môže byť snaha o dosiahnutie čo najostrejšieho glitchu, často to nevedie k vysokej úspešnosti útokov. Ak sa vám s aktuálnym nastavením nedarí nájsť žiadne fungujúce glitche, môže stáť za to mierne upraviť hardvérovú konfiguráciu. Napríklad na doske ChipWhisperer Lite (verzia z 1 časti) môžete odspájkovať prepojku SJ5 a pripájkovať pinovú lištu na JP6. Už len samotné prepojenie týchto pinov pomocou jumpera sa bude správať inak ako napevno zapájkovaná prepojka SJ5.

Viac informácií o hľadaní optimálnych nastavení a o teórii napäťového glitchovania nájdete v prednáškových prezentáciách.

### Praktické cvičenie

Aby sme vás uviedli do problematiky napäťového glitchovania a pomohli vám nájsť vhodné nastavenia, vrátime sa k cvičeniu s glitchovaním v slučke (clock glitching loop). Na začiatku experimentovania môže byť užitočné zaznamenať niekoľko výkonových stôp (power traces), aby ste videli, aký vplyv majú rôzne šírky glitchov na priebeh napájania. Taktiež pamätajte na to, že cieľové mikrokontroléry zvyčajne netolerujú pokles napätia na pine Vcc po dlhšiu dobu bez toho, aby spadli (crash) – akonáhle začne cieľ padať, pri väčších šírkach už nič iné nedosiahnete.

Jedna z vecí, ktorú budete musieť pravdepodobne zmeniť, je parameter opakovania (`repeat`). V závislosti od šírky vášho glitchu sa napätie na napájacom pine nemusí stihnúť zotaviť do momentu, kedy je vložený ďalší glitch. To môže mať za následok zvýšenie sily nasledujúcich glitchov, čo môže (ale nemusí) byť žiaduce. Keďže glitche vložené s parametrom repeat > 1 majú rôznu intenzitu, je dobré skenovať aj rozsah parametra `ext_offset`.

### Glitchovanie na vyšších frekvenciách

Cieľová architektúra XMEGA a v menšej miere aj STM32F3 sa s predvolenými nastaveniami ChipWhispereru glitchujú len veľmi ťažko. Skúste zvýšiť frekvenciu hodín na 24 MHz pre STM32 alebo na 32 MHz pre XMEGA, nastavte parameter repeat na 5-10 a aktivujte high-power aj low-power glitch súčasne. Prenosovú rýchlosť (baud rate) budete musieť upraviť v rovnakom pomere ako hodiny. Tým sa zvýši presnosť glitchu, keďže krok šírky/posunu (width/offset step) sa odvíja od hodín cieľa, čo však môže zároveň znížiť stabilitu cieľového zariadenia.

Ďalšia konfigurácia, ktorá sa osvedčila pri čipe XMEGA, je: odspájkované SJ5, premostené JP6 pomocou jumpera, aktívny high + low power glitch, frekvencia 32 MHz a repeat=5.

ChipWhisperer Husky so svojím glitchovacím systémom založeným na PLL dokáže zvyčajne úspešne glitchovať ciele aj bez nutnosti meniť ich pracovnú frekvenciu.

### Vypnutie logovania

Pri glitchovaní (alebo pri bežnom behu v predchádzajúcich cvičeniach) ste si mohli všimnúť rôzne varovania od logovacích nástrojov, ktoré ChipWhisperer používa. Tie často obsahujú užitočné informácie, najmä ak niečo nefunguje správne, no konkrétne pri napäťovom glitchovaní väčšinou iba zahlcujú textový výstup. Z tohto dôvodu pre cvičenia s napäťovým glitchovaním logovanie vypneme:

```python
cw.set_all_log_levels(cw.logging.CRITICAL)
```

Logovanie môžete znova zapnúť pomocou:

```python
cw.set_all_log_levels(cw.logging.WARNING)
```

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
SS_VER = 'SS_VER_2_1'

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = "../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER=="SS_VER_2_1":
    target.reset_comms()

In [ ]:
def reboot_flush():
    reset_target(scope)
    target.flush()

if scope._is_husky is False:
    if PLATFORM == "CWLITEXMEGA":
        scope.clock.clkgen_freq = 32E6
        if SS_VER=='SS_VER_2_1':
            target.baud = 230400*32/7.37
        else:
            target.baud = 38400*32/7.37
    elif (PLATFORM == "CWLITEARM") or ("F3" in PLATFORM):
        scope.clock.clkgen_freq = 24E6
        if SS_VER=='SS_VER_2_1':
            target.baud = 230400*24/7.37
        else:
            target.baud = 38400*24/7.37
        time.sleep(0.1)

In [ ]:
reboot_flush()
scope.arm()
target.simpleserial_write("g", bytearray([]))
scope.capture()
val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

V ďalšom kroku pripravíme glitch. Hlavné rozdiely v porovnaní s hodinovým glitchovaním sú použitie parametrov `scope.io.glitch_lp/hp`, ktoré aktivujú tranzistory určené pre napäťové pulzy, a nastavenie `scope.glitch.output = 'glitch_only'`, ktoré smeruje výstup glitchu na tranzistory, namiesto vykonania operácie XOR s hodinovým signálom.

In [ ]:
if scope._is_husky:
    scope.glitch.enabled = True
    scope.glitch.clk_src = "pll"
    scope.io.glitch_hp = False
    scope.io.glitch_hp = True
    scope.io.glitch_lp = False
    scope.io.glitch_lp = False
else:
    scope.glitch.clk_src = "clkgen" # set glitch input clock
scope.glitch.output = "glitch_only" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called
if PLATFORM == "CWLITEXMEGA":
    scope.io.glitch_lp = True
    scope.io.glitch_hp = True
elif PLATFORM == "CWLITEARM":
    scope.io.glitch_lp = True
    scope.io.glitch_hp = True
elif PLATFORM == "CW308_STM32F3":
    scope.io.glitch_hp = True
    scope.io.glitch_lp = True

Niekoľko tipov na nájdenie efektívnych glitchov:

1. Tentoraz proti nášmu glitchu bojuje veľa faktorov – na rozdiel od hodinovej línie by napájacia vetva Vcc nemala kmitať! Kratšie glitche preto nebudú mať žiadny efekt. Vhodné šírky sa často nachádzajú tesne pod hranicou, kedy cieľ začne konzistentne padať. Dobrou stratégiou je nájsť minimálnu šírku, pri ktorej dochádza k neustálym pádom, a potom túto šírku mierne zmenšiť.
1. Parameter repeat sa správa úplne inak ako pri hodinovom glitchovaní – pri zvýšenej frekvencii hodín sa napätie Vcc často nestihne zotaviť pred nasledujúcim glitchom. Vyskúšajte preto rôzne hodnoty opakovania.
1. Do glitchovacej slučky sme zabudovali vyhodnocovanie úspechu a resetu. Keď nájdete nejaké potenciálne body pre vloženie chyby, toto vyhodnotenie vám pomôže určiť, ktoré z nich sú pre váš cieľ najlepšie.

Prehľadanie celého vyhľadávacieho priestoru môže trvať veľmi dlho, takže sa môžete rozhodnúť proces zastaviť po dosiahnutí určitého počtu úspechov. Predvolene je tu nastavená hodnota 1, ale môžete ju zmeniť na 10, 20 alebo ísť ešte vyššie.

In [ ]:
MAX_SUCCESSES = 1

In [ ]:
import struct

#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)

# These width/offset settings are for CW-Lite/Pro; width/offset are expressed differently for Husky (see Fault 1_1)
num_tries = 5

gc.set_range("tries", 0, num_tries)
gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step(???)
gc.set_step("tries", 1)
gc.set_step("ext_offset", 1)


scope.adc.timeout = 0.5

reboot_flush()
loff = scope.glitch.offset
lwid = scope.glitch.width
total_successes = 0
successes = 0
resets = 0
for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    scope.glitch.ext_offset = glitch_setting[2]
    if glitch_setting[3] == 0:
        total_successes += successes
        successes = 0
        resets = 0
        if total_successes > MAX_SUCCESSES:
            break
    target.flush()
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        #print("Trigger still high!")
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
        resets += 1

    scope.arm()

    #Do glitch loop
    target.simpleserial_write("g", bytearray([]))

    ret = scope.capture()

    scope.io.vglitch_reset()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")
        resets += 1

        #Device is slow to boot?
        reboot_flush()

    else:
        val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10, timeout=50)#For loop check
        if ???:
            gc.add("reset")
            reboot_flush()
            resets += 1
            #print(val)
        else:
            gcnt = struct.unpack("<I", val['payload'])[0]

            if gcnt != 2500: #for loop check
                gc.add("success")
                print((scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset))
                successes += 1
            else:
                gc.add("normal")
    if (successes > 0):
        print("successes = {}, resets = {}, offset = {}, width = {}, ext_offset = {}".format(successes, resets, scope.glitch.offset, scope.glitch.width, scope.glitch.ext_offset))
        total_successes += successes
print("Done glitching")
#enable logging
cw.set_all_log_levels(cw.logging.WARNING)

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    import struct

#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)

#These width/offset settings are for CW-Lite/Pro; width/offset are expressed differently for Husky (see Fault 1_1)
num_tries = 3
gc.set_range("tries", 1, num_tries)
gc.set_range("width", 43.5, 47.8)
gc.set_range("offset", -48, -10)
gc.set_range("ext_offset", 7, 10)
gc.set_global_step(0.4)
gc.set_step("ext_offset", 1)
scope.glitch.repeat = 11
gc.set_step("tries", 1)



scope.adc.timeout = 0.5

reboot_flush()
loff = scope.glitch.offset
lwid = scope.glitch.width
total_successes = 0
successes = 0
resets = 0
for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    scope.glitch.ext_offset = glitch_setting[2]
    if glitch_setting[3] == 0:
        total_successes += successes
        successes = 0
        resets = 0
        if total_successes > MAX_SUCCESSES:
            break
    target.flush()
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        #print("Trigger still high!")
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
        resets += 1

    scope.arm()

    #Do glitch loop
    target.simpleserial_write("g", bytearray([]))

    ret = scope.capture()

    scope.io.vglitch_reset()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")
        resets += 1

        #Device is slow to boot?
        reboot_flush()

    else:
        val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10, timeout=50)#For loop check
        if val['valid'] is False:
            gc.add("reset")
            reboot_flush()
            resets += 1
            #print(val)
        else:
            gcnt = struct.unpack("<I", val['payload'])[0]

            if gcnt != 2500: #for loop check
                gc.add("success")
                print((scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset))
                successes += 1
            else:
                gc.add("normal")
    if (successes > 0):
        print("successes = {}, resets = {}, offset = {}, width = {}, ext_offset = {}".format(successes, resets, scope.glitch.offset, scope.glitch.width, scope.glitch.ext_offset))
        total_successes += successes
print("Done glitching")
#enable logging
cw.set_all_log_levels(cw.logging.WARNING)
</code></pre>
</details>

Keď je to hotové, vytlačíme naše výsledky. Pravdepodobne budete chcieť ignorovať parametre "tries" a "ext_offset":

In [ ]:
results = gc.calc(ignore_params=["tries", "ext_offset"], sort="success_rate")
results

In [ ]:
scope.dis()
target.dis()